[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_33_Blackboard_Architectures_for_Collaborative_Agents.ipynb)

# Lesson 33 — Blackboard Architectures for Collaborative Agents
**Track 2 · Multi-Agent Coordination · 2 of 5**

> *"A group of specialists is gathered around a blackboard to solve a problem.
> Each watches the board, and whenever they see something they can contribute to,
> they step up and write."* — Erman et al., HEARSAY-II (1980)

### What you'll learn
1. **Why point-to-point A2A breaks** when N agents all need to read each other's work — and why a *shared scratchpad* is a different topology, not just a bigger inbox.
2. **The blackboard pattern** (1970s AI classic, still the right answer for open-ended collaboration): shared structured state + *Knowledge Sources* + a *Controller*.
3. **Hands-on**: build a 4-agent research blackboard — `Searcher`, `Synthesizer`, `Critic`, `Editor` — with a strict Pydantic `Blackboard` state, a typed `Action` log, and a controller that decides who runs next.
4. **A2A ⇄ Blackboard** decision matrix — when each pattern wins, and how to combine them (each Knowledge Source can be an A2A agent under the hood).
5. **Pitfalls** that bite once you go past 2 agents: write conflicts, infinite contribution loops, stale reads, monopoly KSs, version drift.

### Prerequisites
- **Lesson 6** — Multi-agent systems (sequential / orchestrator / critic loop)
- **Lesson 32** — A2A Protocol & Agent-to-Agent Messaging
- A working `ANTHROPIC_API_KEY` in Colab Secrets (`anthropic-api-key`)

### What you ship
A self-contained Colab notebook that runs a 4-agent collaborative research session end-to-end, with:
- a typed shared state (`Blackboard`),
- a deterministic controller,
- a versioned write log,
- and a final assembled briefing — all in one process, no servers.

> Reuse note: the `Blackboard` + `KnowledgeSource` + `Controller` triad here is the foundation we'll extend in L34 (debate w/ judge) and L35 (parallel map-reduce). Keep this notebook open as a reference.

## 1 · Why pairwise A2A breaks past 2 agents

In L32 we shipped real A2A: one agent POSTs a `Task` to another, polls or streams the result. That works beautifully for **request/response** chains — a 2-agent critic loop, a chain of 3 specialists. But A2A is *point-to-point*: every conversation is between exactly two endpoints.

What happens at N=4?

If a `Researcher`, `Synthesizer`, `Critic`, and `Editor` all need to see and react to each other's intermediate work, point-to-point quickly becomes either:

- **A combinatorial mess of endpoints** — every agent has to know every other agent's URL, schema, and lifecycle. Adding a 5th agent means *every other agent* needs an update.
- **A bottleneck through an orchestrator** — one parent agent fans in/out to everyone. But now the orchestrator is the dumb router *and* the smart strategist. And every piece of intermediate work flows back through it, even when two specialists could just read each other's notes.

Concretely, the conversation count blows up:

| N agents | Point-to-point edges | Group-style channels |
|---:|---:|---:|
| 2 | 1 | 1 |
| 3 | 3 | 1 |
| 4 | 6 | 1 |
| 5 | 10 | 1 |
| 6 | 15 | 1 |

Each "edge" above is a separate schema you have to version, a separate retry policy, a separate timeout to tune. **That's why the blackboard exists.**

> 💡 **Mental model**: A2A is *email*. Blackboard is *a shared Google Doc the whole team has open*. Email is great for "send this to that person." Google Docs is great for "the team is co-writing something." Use the right tool — and yes, they compose (you can still email someone *about* the doc).

## 2 · The blackboard pattern (HEARSAY-II, 1971–1976)

The pattern comes from **HEARSAY-II**, a speech-understanding system built at CMU under ARPA's Speech Understanding Research program. The problem: turn a waveform into a sentence. The waveform is noisy; competing hypotheses live at the *signal*, *phoneme*, *word*, and *phrase* levels simultaneously.

HEARSAY-II's insight: **don't pick a fixed pipeline.** Different specialists are good at different things:
- one is great at finding word boundaries,
- one is great at proposing phonemes from acoustic features,
- one is great at scoring whether a proposed word *fits* a partial sentence.

So instead of forcing them into a fixed order, the system gave them a **shared structured workspace** — the blackboard — partitioned by abstraction level. Each specialist watched the board; when one saw something it could contribute to, it would write a *hypothesis* back. A separate **scheduler** decided who got to run next.

The three load-bearing pieces:

1. **The Blackboard** — a *structured*, *versioned*, *shared* representation of the problem state. Not a free-form chat log — typed slots: "current word hypotheses at time 0.4s", "current parse tree at level 3", etc.
2. **Knowledge Sources (KSs)** — specialists. Each one has (a) a *condition* describing what kind of blackboard state activates it, and (b) an *action* that contributes new entries when invoked.
3. **The Controller / Scheduler** — looks at the board, looks at which KSs *want* to run, and picks one (or several). This is where strategy lives: depth-first, breadth-first, follow-the-best-hypothesis, etc.

This is exactly what we want for open-ended LLM agent collaboration. The "specialists" become Anthropic-API-backed agents. The "blackboard" becomes a Pydantic model. The "scheduler" becomes a tiny Python loop with rules.

## 3 · Designing our shared state

We're going to build a **collaborative research briefing** system. The team's goal: produce a 5–7 bullet briefing on a topic, well-supported, criticized, then declared done.

Concretely we'll have four Knowledge Sources:

| KS | What it watches for | What it writes |
|---|---|---|
| **Searcher** | `sources` is empty or too small | `Source` records (title, snippet) |
| **Synthesizer** | enough `sources`, not enough `bullets` | `Bullet` records grounded in sources |
| **Critic** | unreviewed bullets exist | `Issue` records pointing at specific bullets |
| **Editor** | bullets exist, no open critical issues | sets `done=True`, writes `final_summary` |

The blackboard itself is one Pydantic model. Three rules:

1. **Append-only for content** — Searcher/Synthesizer/Critic *add* rows; they don't overwrite each other. Editor is the only KS that can flip `done`.
2. **Every write bumps `version`** — the controller and KSs can detect "did anything change last turn?"
3. **Every write also appends an `Action` to a history log** — gives us a perfectly reproducible audit trail (great for the L24 reliability harness).

Let's install dependencies and define the state.

In [ ]:
# ── Setup: install deps and load the API key ──────────────────────────────────
!pip install anthropic pydantic -q

import os
try:
    from google.colab import userdata  # type: ignore
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("anthropic-api-key")
    print("✓ ANTHROPIC_API_KEY loaded from Colab Secrets")
except Exception:
    assert os.environ.get("ANTHROPIC_API_KEY"), (
        "Set ANTHROPIC_API_KEY in your env, or add it to Colab Secrets as 'anthropic-api-key'."
    )
    print("✓ ANTHROPIC_API_KEY found in environment")

from anthropic import Anthropic
client = Anthropic()
HAIKU = "claude-haiku-4-5"
SONNET = "claude-sonnet-4-5"
print("Anthropic client ready.")

### 3.1 · The `Blackboard` itself

Notice we don't put free-form strings on the blackboard — every entry is a typed row. That's the *structured-ness* HEARSAY-II insisted on. Pydantic enforces it.

We also expose two read helpers — `unreviewed_bullets()` and `open_issues()` — so KSs don't have to reach into the internals. Those helpers are the *vocabulary* of the activation conditions.

In [ ]:
from __future__ import annotations

from datetime import datetime
from typing import Literal
from pydantic import BaseModel, Field


# ── Typed rows on the board ───────────────────────────────────────────────────
class Source(BaseModel):
    id: int
    title: str
    snippet: str
    added_by: str            # which KS wrote this
    version_added: int

class Bullet(BaseModel):
    id: int
    text: str
    supports: list[int] = Field(default_factory=list)   # Source ids
    added_by: str
    version_added: int
    reviewed: bool = False    # set True once a Critic has looked at it

class Issue(BaseModel):
    id: int
    bullet_id: int
    severity: Literal["minor", "major", "blocker"]
    description: str
    raised_by: str
    version_added: int
    resolved: bool = False

class Action(BaseModel):
    """Audit-log entry: who did what at which board version."""
    version: int
    actor: str
    kind: Literal["read", "add_source", "add_bullet", "add_issue",
                  "mark_reviewed", "resolve_issue", "set_done", "no_op"]
    detail: str
    at: datetime = Field(default_factory=datetime.utcnow)


# ── The board ─────────────────────────────────────────────────────────────────
class Blackboard(BaseModel):
    topic: str
    version: int = 0
    sources: list[Source] = Field(default_factory=list)
    bullets: list[Bullet] = Field(default_factory=list)
    issues: list[Issue] = Field(default_factory=list)
    done: bool = False
    final_summary: str | None = None
    history: list[Action] = Field(default_factory=list)

    # ── id allocators ────────────────────────────────────────────────────────
    def _next_source_id(self) -> int: return len(self.sources) + 1
    def _next_bullet_id(self) -> int: return len(self.bullets) + 1
    def _next_issue_id(self)  -> int: return len(self.issues)  + 1

    # ── writes (every write bumps version + appends Action) ──────────────────
    def add_source(self, title: str, snippet: str, by: str) -> Source:
        self.version += 1
        s = Source(id=self._next_source_id(), title=title, snippet=snippet,
                   added_by=by, version_added=self.version)
        self.sources.append(s)
        self.history.append(Action(version=self.version, actor=by,
                                   kind="add_source", detail=f"#{s.id} {title!r}"))
        return s

    def add_bullet(self, text: str, supports: list[int], by: str) -> Bullet:
        self.version += 1
        b = Bullet(id=self._next_bullet_id(), text=text, supports=supports,
                   added_by=by, version_added=self.version)
        self.bullets.append(b)
        self.history.append(Action(version=self.version, actor=by,
                                   kind="add_bullet", detail=f"#{b.id} {text[:40]!r}"))
        return b

    def add_issue(self, bullet_id: int, severity: str, description: str, by: str) -> Issue:
        self.version += 1
        i = Issue(id=self._next_issue_id(), bullet_id=bullet_id,
                  severity=severity, description=description,
                  raised_by=by, version_added=self.version)
        self.issues.append(i)
        # mark the bullet reviewed: the Critic actually looked at it
        for b in self.bullets:
            if b.id == bullet_id:
                b.reviewed = True
        self.history.append(Action(version=self.version, actor=by,
                                   kind="add_issue",
                                   detail=f"#{i.id} on bullet#{bullet_id} ({severity})"))
        return i

    def mark_all_reviewed(self, by: str) -> None:
        self.version += 1
        for b in self.bullets:
            b.reviewed = True
        self.history.append(Action(version=self.version, actor=by,
                                   kind="mark_reviewed", detail="all"))

    def resolve_issue(self, issue_id: int, by: str) -> None:
        self.version += 1
        for i in self.issues:
            if i.id == issue_id:
                i.resolved = True
        self.history.append(Action(version=self.version, actor=by,
                                   kind="resolve_issue", detail=f"#{issue_id}"))

    def set_done(self, summary: str, by: str) -> None:
        self.version += 1
        self.done = True
        self.final_summary = summary
        self.history.append(Action(version=self.version, actor=by,
                                   kind="set_done", detail=summary[:60]))

    def log_noop(self, by: str, reason: str) -> None:
        self.history.append(Action(version=self.version, actor=by,
                                   kind="no_op", detail=reason))

    # ── reads (the vocabulary of activation conditions) ─────────────────────
    def unreviewed_bullets(self) -> list[Bullet]:
        return [b for b in self.bullets if not b.reviewed]

    def open_issues(self, min_severity: str = "major") -> list[Issue]:
        order = {"minor": 0, "major": 1, "blocker": 2}
        thr = order[min_severity]
        return [i for i in self.issues if not i.resolved and order[i.severity] >= thr]

print("Blackboard model ready.")

## 4 · Knowledge Sources — the interface

Every KS gets the *same* shape:

```python
class KnowledgeSource:
    name: str
    def can_contribute(self, board: Blackboard) -> bool: ...
    def contribute(self, board: Blackboard) -> None: ...
```

- `can_contribute(board)` is the **activation condition** — pure read. Cheap. No LLM calls.
- `contribute(board)` is the **action** — what the KS writes back. *This* is where the LLM call happens.

Splitting condition from action is the whole reason the blackboard pattern scales: the scheduler can ask N specialists "can you contribute right now?" without paying N LLM calls. Only the *chosen* one actually pays.

In [ ]:
import json as _json
from typing import Protocol


class KnowledgeSource(Protocol):
    """Anything with a name, an activation predicate, and a contribute method."""
    name: str
    def can_contribute(self, board: Blackboard) -> bool: ...
    def contribute(self, board: Blackboard) -> None: ...


# ── Small Anthropic helper that forces a JSON tool call ──────────────────────
def call_with_tool(system: str, user: str, tool: dict, model: str = HAIKU) -> dict:
    """Send one user message; require the model to call `tool` and return its input dict."""
    resp = client.messages.create(
        model=model,
        max_tokens=1024,
        system=system,
        messages=[{"role": "user", "content": user}],
        tools=[tool],
        tool_choice={"type": "tool", "name": tool["name"]},
    )
    for block in resp.content:
        if block.type == "tool_use" and block.name == tool["name"]:
            return block.input
    raise RuntimeError(f"Model did not call required tool {tool['name']}")

print("KnowledgeSource protocol + tool-call helper ready.")

## 5 · Specialist 1 — `Searcher`

*Activation:* fires when we have fewer than `min_sources` rows on the board.

*Action:* asks Haiku to propose a small batch of plausible "sources" (we're simulating real search here — in production you'd plug in Tavily, Brave, or your internal corpus from L20). Note we mark every Source with `added_by=self.name` so the audit trail is meaningful.

In [ ]:
SEARCHER_TOOL = {
    "name": "propose_sources",
    "description": "Propose 2–3 candidate sources for the research topic. "
                   "Sources must look like real reference snippets (no fabricated URLs).",
    "input_schema": {
        "type": "object",
        "properties": {
            "sources": {
                "type": "array",
                "minItems": 2, "maxItems": 3,
                "items": {
                    "type": "object",
                    "properties": {
                        "title": {"type": "string"},
                        "snippet": {"type": "string"},
                    },
                    "required": ["title", "snippet"],
                },
            }
        },
        "required": ["sources"],
    },
}

class Searcher:
    name = "Searcher"
    def __init__(self, min_sources: int = 4):
        self.min_sources = min_sources

    # ── condition ─────────────────────────────────────────────────────────
    def can_contribute(self, board: Blackboard) -> bool:
        return len(board.sources) < self.min_sources

    # ── action ────────────────────────────────────────────────────────────
    def contribute(self, board: Blackboard) -> None:
        existing = "\n".join(f"- {s.title}" for s in board.sources) or "(none yet)"
        prompt = (
            f"Topic: {board.topic}\n\n"
            f"Existing sources on the board:\n{existing}\n\n"
            "Propose 2–3 NEW reference snippets the team should consider. "
            "Each snippet should be a short factual paragraph (1–3 sentences). "
            "Do not repeat existing sources."
        )
        out = call_with_tool(
            system=("You are a Researcher writing concise factual reference snippets. "
                    "Never fabricate URLs or fake citations — just title + snippet."),
            user=prompt,
            tool=SEARCHER_TOOL,
        )
        for s in out["sources"]:
            board.add_source(title=s["title"], snippet=s["snippet"], by=self.name)

print("Searcher KS ready.")

## 6 · Specialist 2 — `Synthesizer`

*Activation:* fires when we have enough sources but not yet enough bullets, **and** the latest blackboard write wasn't from another Synthesizer (else we'd loop on ourselves).

*Action:* drafts bullets grounded in existing source ids. We force the model to attach `supports: [source_ids]` — this is the L18-style citation discipline carried into a multi-agent setting.

In [ ]:
SYNTH_TOOL = {
    "name": "propose_bullets",
    "description": "Draft new bullets for the briefing. Each bullet MUST be supported by ≥1 existing source id.",
    "input_schema": {
        "type": "object",
        "properties": {
            "bullets": {
                "type": "array", "minItems": 1, "maxItems": 4,
                "items": {
                    "type": "object",
                    "properties": {
                        "text": {"type": "string"},
                        "supports": {"type": "array", "items": {"type": "integer"}, "minItems": 1},
                    },
                    "required": ["text", "supports"],
                }
            }
        },
        "required": ["bullets"],
    },
}

class Synthesizer:
    name = "Synthesizer"
    def __init__(self, target_bullets: int = 6):
        self.target = target_bullets

    def can_contribute(self, board: Blackboard) -> bool:
        if len(board.sources) < 3:                  # need enough material first
            return False
        if len(board.bullets) >= self.target:        # already have enough
            return False
        # Avoid running back-to-back: let Critic see what we wrote.
        if board.history and board.history[-1].actor == self.name:
            return False
        return True

    def contribute(self, board: Blackboard) -> None:
        sources_md = "\n".join(f"[{s.id}] {s.title}: {s.snippet}" for s in board.sources)
        bullets_md = "\n".join(f"- {b.text}" for b in board.bullets) or "(none yet)"
        prompt = (
            f"Topic: {board.topic}\n\n"
            f"Available sources (cite by id):\n{sources_md}\n\n"
            f"Existing bullets:\n{bullets_md}\n\n"
            f"Add {self.target - len(board.bullets)} new bullets. "
            "Each bullet must cite ≥1 source id in `supports`."
        )
        out = call_with_tool(
            system=("You synthesize crisp briefing bullets from sources. "
                    "Never write a bullet without a supporting source id."),
            user=prompt,
            tool=SYNTH_TOOL,
        )
        valid_source_ids = {s.id for s in board.sources}
        for b in out["bullets"]:
            supports = [sid for sid in b["supports"] if sid in valid_source_ids]
            if not supports:        # drop hallucinated source ids
                continue
            board.add_bullet(text=b["text"], supports=supports, by=self.name)

print("Synthesizer KS ready.")

## 7 · Specialist 3 — `Critic`

*Activation:* fires when there is at least one **unreviewed** bullet on the board.

*Action:* picks the unreviewed bullets, raises typed `Issue` rows (severity minor / major / blocker), and *also* marks bullets as reviewed even if no issue was found (`mark_all_reviewed` is too aggressive — we only mark the ones we actually critiqued, which `add_issue` already does row-by-row).

In [ ]:
CRITIC_TOOL = {
    "name": "raise_issues",
    "description": "For each bullet you reviewed, EITHER raise an issue OR explicitly say it's fine. Return one entry per bullet reviewed.",
    "input_schema": {
        "type": "object",
        "properties": {
            "reviews": {
                "type": "array", "minItems": 1,
                "items": {
                    "type": "object",
                    "properties": {
                        "bullet_id": {"type": "integer"},
                        "is_fine": {"type": "boolean"},
                        "severity": {"type": "string",
                                     "enum": ["minor", "major", "blocker"]},
                        "issue": {"type": "string"},
                    },
                    "required": ["bullet_id", "is_fine"],
                },
            }
        },
        "required": ["reviews"],
    },
}

class Critic:
    name = "Critic"

    def can_contribute(self, board: Blackboard) -> bool:
        return len(board.unreviewed_bullets()) > 0

    def contribute(self, board: Blackboard) -> None:
        targets = board.unreviewed_bullets()
        targets_md = "\n".join(
            f"#{b.id}: {b.text}  (supports={b.supports})"
            for b in targets
        )
        sources_md = "\n".join(f"[{s.id}] {s.title}: {s.snippet}" for s in board.sources)
        prompt = (
            f"Topic: {board.topic}\n\n"
            f"Sources on the board:\n{sources_md}\n\n"
            f"Review every bullet below. For each, decide if it is fine or has an issue.\n"
            f"Common issues: unsupported claim, overstatement, vague language, missing nuance, contradicts a source.\n\n"
            f"Bullets to review:\n{targets_md}"
        )
        out = call_with_tool(
            system=("You are a strict but fair reviewer. Only raise an issue if there is a concrete problem. "
                    "Resist nitpicking; prefer 'is_fine: true' when the bullet is decent."),
            user=prompt,
            tool=CRITIC_TOOL,
        )
        target_ids = {b.id for b in targets}
        for r in out["reviews"]:
            bid = r["bullet_id"]
            if bid not in target_ids:           # ignore hallucinated bullet ids
                continue
            if r["is_fine"]:
                # mark reviewed without raising an issue: append a no-op-ish "minor / OK" record
                # by bumping version & adding a benign history line (no Issue row)
                board.version += 1
                for b in board.bullets:
                    if b.id == bid:
                        b.reviewed = True
                board.history.append(Action(
                    version=board.version, actor=self.name,
                    kind="mark_reviewed", detail=f"bullet#{bid} OK"))
            else:
                board.add_issue(bullet_id=bid,
                                severity=r.get("severity", "minor"),
                                description=r.get("issue", "(no description)"),
                                by=self.name)

print("Critic KS ready.")

## 8 · Specialist 4 — `Editor`

*Activation:* fires when:
- we have at least `min_bullets` bullets,
- all bullets have been reviewed,
- there are no open issues at `major` or `blocker` severity,
- and we're not already `done`.

*Action:* drafts the final briefing summary and sets `done=True`. **This is the only KS allowed to flip `done`.** That single privilege is what stops the system from looping forever.

> 💡 **Design pattern**: in every multi-agent system, *exactly one* role should own the termination decision. If two agents can both decide "we're finished," they will disagree, and you'll either get premature stops or infinite loops.

In [ ]:
EDITOR_TOOL = {
    "name": "finalize_briefing",
    "description": "Write a tight 3–5 sentence executive summary covering all the bullets.",
    "input_schema": {
        "type": "object",
        "properties": {"summary": {"type": "string"}},
        "required": ["summary"],
    },
}

class Editor:
    name = "Editor"
    def __init__(self, min_bullets: int = 5):
        self.min_bullets = min_bullets

    def can_contribute(self, board: Blackboard) -> bool:
        if board.done:
            return False
        if len(board.bullets) < self.min_bullets:
            return False
        if board.unreviewed_bullets():
            return False
        if board.open_issues(min_severity="major"):
            return False
        return True

    def contribute(self, board: Blackboard) -> None:
        bullets_md = "\n".join(f"- {b.text}" for b in board.bullets)
        prompt = (
            f"Topic: {board.topic}\n\n"
            f"Approved bullets:\n{bullets_md}\n\n"
            "Write a 3–5 sentence executive summary that weaves these together. "
            "No new claims; only synthesize what's already approved."
        )
        out = call_with_tool(
            system=("You are an Editor. You synthesize approved bullets into a tight executive summary. "
                    "You do NOT introduce new facts."),
            user=prompt,
            tool=EDITOR_TOOL,
        )
        board.set_done(summary=out["summary"], by=self.name)

print("Editor KS ready.")

## 9 · The Controller / Scheduler

This is where the *strategy* of the system lives. Possible schedules:

| Strategy | How it picks | When to use |
|---|---|---|
| **Fixed priority** | iterate KSs in a fixed order, pick the first that can contribute | simple, deterministic, our default |
| **Random** | among willing KSs, pick uniformly | useful for fairness / exploration |
| **Score-based** | each KS reports a "utility" for running now; pick highest | classic HEARSAY-II move |
| **Parallel** | run *all* willing KSs concurrently (next lesson, L35) | fan-out is independent |

We'll do **fixed priority** — Editor > Critic > Synthesizer > Searcher. Why that order? Always prefer the closer-to-termination KS. Editor wraps things up if it can; otherwise Critic resolves outstanding work; otherwise Synthesizer adds bullets; only if all else fails does Searcher fetch more material. This naturally avoids the "infinite searching" failure mode.

Notice the **termination conditions** — three of them, OR-ed together:
- `board.done` (Editor said so),
- `max_steps` (hard upper bound — *always* have one),
- *no KS willing to contribute* (deadlock, surface it loudly).

In [ ]:
class Controller:
    """Fixed-priority scheduler over a list of KnowledgeSources."""

    def __init__(self, knowledge_sources: list, max_steps: int = 25):
        # Note: the order of this list IS the priority — first willing KS wins.
        self.knowledge_sources = knowledge_sources
        self.max_steps = max_steps

    def run(self, board: Blackboard, verbose: bool = True) -> Blackboard:
        for step in range(1, self.max_steps + 1):
            if board.done:
                if verbose: print(f"✓ done at step {step-1}")
                return board

            chosen = next((ks for ks in self.knowledge_sources if ks.can_contribute(board)), None)
            if chosen is None:
                if verbose: print(f"✗ deadlock at step {step}: no KS willing to contribute.")
                board.log_noop(by="Controller", reason="no KS willing")
                return board

            if verbose:
                print(f"[step {step:>2}] picked {chosen.name}  (board.version={board.version})")
            try:
                chosen.contribute(board)
            except Exception as e:
                if verbose: print(f"   ! {chosen.name} raised {type(e).__name__}: {e}")
                board.log_noop(by=chosen.name, reason=f"error: {type(e).__name__}")

        if verbose: print(f"✗ stopped at max_steps={self.max_steps} without DONE")
        return board

print("Controller ready.")

## 10 · Live run — collaborative briefing

Wire up all four KSs and run the controller on a real topic. Watch the trace — you'll see Searcher front-load, Synthesizer take over once sources exist, Critic interleave reviews, and Editor only fire at the very end.

> 💡 **EXPERIMENT**: Try (a) changing the priority order to put `Synthesizer` ahead of `Critic`, and watch the bullet count overshoot. (b) Drop the `Critic` entirely and see how `Editor`'s activation never fires (because bullets stay unreviewed). The activation conditions are the *contract* between KSs.

In [ ]:
board = Blackboard(topic="The 2010s deep-learning revolution: what unlocked it and what it didn't solve")

ctrl = Controller(
    knowledge_sources=[
        Editor(min_bullets=5),           # highest priority: wrap up if possible
        Critic(),                        # then review what's been written
        Synthesizer(target_bullets=6),   # then add bullets
        Searcher(min_sources=4),         # only search if we still need material
    ],
    max_steps=25,
)

board = ctrl.run(board, verbose=True)

### 10.1 · Inspect the board

After the run, the board is a complete record. We have the sources, the bullets, the issues, and the audit log of who did what at which version.

In [ ]:
import textwrap

print(f"Topic: {board.topic}")
print(f"Final version: {board.version}    done: {board.done}")
print(f"Sources: {len(board.sources)}  Bullets: {len(board.bullets)}  Issues: {len(board.issues)}")
print()
print("── Sources ─────────────────────────────────────────")
for s in board.sources:
    print(f"[{s.id}] ({s.added_by}) {s.title}")
    print(textwrap.fill(f"      {s.snippet}", width=92, subsequent_indent="      "))
print()
print("── Bullets ─────────────────────────────────────────")
for b in board.bullets:
    flag = "✓" if b.reviewed else "✗"
    print(f"{flag} #{b.id} ({b.added_by}, supports={b.supports}): {b.text}")
print()
print("── Issues ──────────────────────────────────────────")
for i in board.issues:
    state = "resolved" if i.resolved else "open"
    print(f"#{i.id} on bullet#{i.bullet_id} [{i.severity}/{state}] ({i.raised_by}): {i.description}")
print()
print("── Final summary ───────────────────────────────────")
print(textwrap.fill(board.final_summary or "(not finalized)", width=92))

### 10.2 · Replay the audit log

This is the killer feature of the structured blackboard: the `history` log is a perfect *causal record*. You can replay it, diff two runs of the same topic, or use it as ground-truth for the L24 reliability harness.

In [ ]:
for a in board.history:
    print(f"v{a.version:>2}  {a.actor:<12}  {a.kind:<14}  {a.detail}")

## 11 · A2A vs Blackboard — decision matrix

| Aspect | **A2A (L32)** | **Blackboard (this lesson)** |
|---|---|---|
| Topology | point-to-point | shared scratchpad + N readers/writers |
| Best for | request/response chains, hand-offs | open-ended collaboration, iterative refinement |
| Number of agents | scales linearly up to ~5 | scales to ~10 before scheduler complexity bites |
| Coordination locus | embedded in the conversation (who calls whom) | externalized in the controller |
| State | inside each Task object | one source of truth on the board |
| Process model | one process per agent (typically) | one process, multiple specialists (or A2A KSs) |
| Failure mode if you pick wrong | combinatorial endpoint sprawl | dumb controller turning the board into a thrash log |
| Termination | task lifecycle (`completed` / `failed`) | controller observing board state |

**Compose them.** The blackboard *controller* is local. Each *Knowledge Source* can be:
- a plain Python class (what we built),
- a function calling Haiku (what we built),
- **an A2A agent at the other end of HTTP** (next section).

That last option is the production answer: blackboard for coordination, A2A for distributed execution.

## 12 · Optional — wrapping a KS as an A2A client

We don't run a fresh A2A server here (we built that in L32), but here is the *exact* shape a `KnowledgeSource` would take if `Synthesizer` lived as an A2A agent at `http://synth.example/`.

The blackboard pattern doesn't care where the specialist runs. The controller still sees only `name + can_contribute + contribute`.

```python
import httpx, uuid

class A2ASynthesizer:
    name = "Synthesizer"
    def __init__(self, base_url: str, target_bullets: int = 6, poll_interval: float = 0.3):
        self.base_url = base_url.rstrip("/")
        self.target = target_bullets
        self.poll_interval = poll_interval

    def can_contribute(self, board: Blackboard) -> bool:
        # exactly the same predicate as the local Synthesizer
        return (
            len(board.sources) >= 3
            and len(board.bullets) < self.target
            and (not board.history or board.history[-1].actor != self.name)
        )

    def contribute(self, board: Blackboard) -> None:
        payload = {
            "id": str(uuid.uuid4()),                    # idempotency key (L32 pitfall #1)
            "message": {
                "parts": [
                    {"type": "DataPart", "data": board.model_dump()},  # send the WHOLE board
                ]
            }
        }
        r = httpx.post(f"{self.base_url}/tasks/send", json=payload, timeout=30).json()
        task_id = r["id"]
        # poll till done (or use SSE stream)
        while True:
            t = httpx.get(f"{self.base_url}/tasks/{task_id}", timeout=10).json()
            if t["status"]["state"] in {"completed", "failed", "canceled"}:
                break
            time.sleep(self.poll_interval)
        # take the artifact (a list of {text, supports}) and write to the board
        for art in t.get("artifacts", []):
            for part in art["parts"]:
                if part["type"] == "DataPart":
                    for b in part["data"].get("bullets", []):
                        board.add_bullet(text=b["text"], supports=b["supports"], by=self.name)
```

That's *all* it takes to lift a specialist out of process. The pattern survives the transport boundary unchanged.

> 💡 **The big realization**: a controller talking to local objects, in-process functions, and remote A2A agents all look the same to the controller. That's why we put effort into a typed `Blackboard` and a tight `KnowledgeSource` protocol.

## 13 · Pitfalls (the 10 that will bite you)

1. **The single-decider rule.** Exactly one KS should own termination. We gave Editor that privilege. Two KSs both able to flip `done` → premature stop or no stop at all.
2. **Back-to-back same-KS firing.** Without the "don't fire if I just fired" guard on Synthesizer, a hot KS monopolizes the board. Always check `board.history[-1].actor != self.name` in the activation predicate.
3. **Reading stale board state.** A KS *closes over* a snapshot of the board when its `contribute()` starts, but the board may already be different by the time it tries to write (in the parallel case — L35). Carry a `version_seen` and check on commit.
4. **Hallucinated ids.** The Synthesizer happily cites source #99 that doesn't exist; the Critic happily raises an issue on bullet #42 that doesn't exist. Validate against `{s.id for s in board.sources}` / `{b.id for b in board.bullets}` *before* writing.
5. **Activation condition cost.** `can_contribute` is supposed to be cheap. If you find yourself making LLM calls inside it, you've inverted the pattern — split the KS into "trigger" and "act."
6. **No deadlock detection.** If *no* KS is willing, the controller has to surface that — silent stalls are the worst kind of bug. We return early with a `no_op` log line.
7. **Fixed priority can starve specialists.** Editor-first sounds good until you realize Searcher never gets a chance to add more sources mid-run. Consider score-based or rotating schedulers in production.
8. **Untyped append-only history.** A free-form text log gives you nothing. A typed `Action` log lets you replay, diff, and reliability-check (L24).
9. **Schema drift across KSs.** The whole point of structured rows is that specialists agree. The moment Synthesizer starts writing extra unspec'd fields, Critic's parser quietly drops them. Pin the schemas in one place (`Blackboard` Pydantic model) and version it.
10. **Picking blackboard when A2A is right.** If your problem is a simple chain — `A → B → C` — a blackboard is overhead. Stay with A2A. Switch to blackboard when *every* specialist plausibly needs to read every other specialist's work.

## 14 · Homework

Choose two; ship what you build to the workspace.

1. **Score-based scheduler.** Replace `Controller` with a `ScoredController` that asks each willing KS for a `priority: float` and picks the max. Make Editor return a sky-high priority once allowed, Critic return `0.7 * (#unreviewed bullets)`, Synthesizer return `0.5 * (target - #bullets)`, Searcher return `0.3 * (min - #sources)`. Run side-by-side with the fixed-priority version on the same topic — compare audit logs.
2. **Add a 5th KS: `Annotator`.** Job: when a bullet has only one supporting source, fetch a second one from existing sources (no Searcher trip needed). Activation: any bullet with `len(supports) == 1` exists and is reviewed.
3. **A2A-ize the Critic.** Run the L32 A2A Critic server (the one we already built) and write `A2ACritic(KnowledgeSource)` that POSTs the board's unreviewed bullets, polls, then writes back `Issue` rows. Confirm the controller code does not change.
4. **Reliability hook.** Add `auto_researcher/reliability/blackboard_slo.py` defining `BlackboardSLO(max_steps_to_done=15, max_open_blockers_at_done=0, min_bullet_review_coverage=1.0)` and a `ci_gate(board)` method. Wire into the L24 reliability harness.
5. **Diff two runs.** Run the same topic with `Critic` enabled vs disabled. Save both `board.history` lists as JSON. Write a `diff_runs(a, b)` function that prints which versions diverged and the average "issue density per bullet" of each.

## 15 · Coming next

**Lesson 34 — Debate Systems with Judge.** Two (or more) agents *argue opposing positions*; a separate Judge agent reads the transcript and decides. Different from the Critic loop: in debate, *both* sides know they're advocates, and the truth-seeking happens through adversarial reasoning rather than one-sided revision. We'll connect it back to today's blackboard — debate is a blackboard with two KSs that produce strictly alternating contributions and a Judge that owns termination.

**Track 2 roadmap** (unchanged):
- L32 ✅ A2A protocol
- **L33 ✅ Blackboard (this lesson)**
- L34 ⏳ Debate with judge
- L35 ⏳ Parallel fan-out / map-reduce across A2A peers
- L36 ⏳ Track 2 capstone — Multi-Agent Research Swarm (A2A + blackboard + debate)

When this notebook runs cleanly end-to-end, you've internalized the third coordination primitive — sequential pipelines (L6), point-to-point messaging (L32), and the shared scratchpad (this one). Everything else in multi-agent design is a remix of these three.